# SynBioCrow 2.4.13 — Development Route Generation
Runs only the four frozen 2.4 development targets. Bakuchiol and D-allitol remain sealed validation targets.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_2_4_13")
OUT=Path("/content/drive/MyDrive/SynBioCrow/2.4/development_routes_2_4_13")
RUNOUT=Path("/content/synbiocrow_2_4_13")
STATE=Path("/content/drive/MyDrive/SynBioCrow/2.4/state_2_4_13")
OUT.mkdir(parents=True,exist_ok=True); STATE.mkdir(parents=True,exist_ok=True)

def run(cmd,cwd=None,timeout=3600,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    env=os.environ.copy()
    if cwd: env["PYTHONPATH"]=str(cwd)+os.pathsep+env.get("PYTHONPATH","")
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode: raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

print("[2.4.13]  5% | clone develop/2.4",flush=True)
if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.4","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=900)
run([sys.executable,"-m","pip","install","-q","doranet==0.5.7a1"],timeout=900,check=False)

print("[2.4.13] 20% | backend readiness",flush=True)
run([sys.executable,"-m","synbiocrow.cli","readiness","--json"],cwd=REPO,timeout=300)

print("[2.4.13] 30% | frozen development-only generation",flush=True)
if RUNOUT.exists(): shutil.rmtree(RUNOUT)
run([sys.executable,str(REPO/"scripts/v24_generate_development_routes.py"),
     "--tranche",str(REPO/"benchmarks/2.4/source_tranche_v1.json"),
     "--split-manifest",str(REPO/"benchmarks/2.4/split_manifest_2_4_11.json"),
     "--out-dir",str(RUNOUT),"--state-dir",str(STATE),
     "--max-route-steps","8","--max-routes","100"],cwd=REPO,timeout=7200)

summary=json.loads((RUNOUT/"generation_summary.json").read_text())
assert summary["validation_truth_accessed"] is False
assert summary["tuning_performed"] is False
assert len(summary["targets"])==4
assert set(summary["sealed_validation_record_ids"])=={"post2022-bakuchiol-2024","post2022-allitol-2025"}

print("[2.4.13] 85% | persist outputs",flush=True)
for p in RUNOUT.iterdir():
    if p.is_file(): shutil.copy2(p,OUT/p.name)
bundle=OUT/"SynBioCrow_2_4_13_DEVELOPMENT_ROUTE_GENERATION_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=bundle: z.write(p,p.name)
print("[2.4.13] 100% | PASS",flush=True)
print("[2.4.13] bundle",bundle,flush=True)
print("[2.4.13] SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest(),flush=True)
files.download(str(bundle))


## Rescue download
Run only if the automatic browser download was missed.


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/development_routes_2_4_13/SynBioCrow_2_4_13_DEVELOPMENT_ROUTE_GENERATION_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
